In [2]:
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load embeddings
X_embeddings = np.load("../models/X_embeddings.npy")

# Load labels
y_labels = np.load("../models/y_labels.npy", allow_pickle=True)

# Load label encoder
label_encoder = joblib.load("../models/label_encoder.pkl")

print("Embeddings shape:", X_embeddings.shape)
print("Number of labels:", len(y_labels))
print("Classes:", label_encoder.classes_)

Embeddings shape: (15000, 320)
Number of labels: 15000
Classes: ['binding' 'enzyme' 'transport']


In [4]:
ensemble_probs = np.load("../models/ensemble_probs.npy")
print("Ensemble probabilities shape:", ensemble_probs.shape)

Ensemble probabilities shape: (3000, 3)


In [6]:
X_test = np.load("../models/X_test_embeddings.npy")
y_test = np.load("../models/y_test_encoded.npy")

print("Test embeddings shape:", X_test.shape)
print("Test labels shape:", y_test.shape)

Test embeddings shape: (3000, 320)
Test labels shape: (3000,)


In [7]:
import numpy as np
import joblib

# Load test embeddings and labels
X_test = np.load("../models/X_test_embeddings.npy")
y_test = np.load("../models/y_test_encoded.npy")

# Load label encoder
label_encoder = joblib.load("../models/label_encoder.pkl")

# Load ensemble probabilities
ensemble_probs = np.load("../models/ensemble_probs.npy")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Classes:", label_encoder.classes_)
print("Ensemble probabilities shape:", ensemble_probs.shape)

X_test shape: (3000, 320)
y_test shape: (3000,)
Classes: ['binding' 'enzyme' 'transport']
Ensemble probabilities shape: (3000, 3)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity among all test proteins
similarity_matrix = cosine_similarity(X_test)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (3000, 3000)


In [9]:
import numpy as np

# Keep only strong similarities
threshold = 0.95
similarity_filtered = similarity_matrix.copy()
similarity_filtered[similarity_filtered < threshold] = 0

# Normalize rows
row_sums = similarity_filtered.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
W = similarity_filtered / row_sums

# Diffuse labels
alpha = 0.7
diffused_probs = alpha * ensemble_probs + (1 - alpha) * W.dot(ensemble_probs)

# Final predictions
y_pred_diffused = np.argmax(diffused_probs, axis=1)

print("Homology-based label diffusion completed.")

Homology-based label diffusion completed.


In [10]:
from sklearn.metrics import accuracy_score, classification_report

# Accuracy
accuracy_diffused = accuracy_score(y_test, y_pred_diffused)

print("Homology-Based Label Diffusion Accuracy:", accuracy_diffused)

print("\nClassification Report:\n")
print(
    classification_report(
        y_test,
        y_pred_diffused,
        target_names=label_encoder.classes_
    )
)

Homology-Based Label Diffusion Accuracy: 0.942

Classification Report:

              precision    recall  f1-score   support

     binding       0.93      0.95      0.94      1000
      enzyme       0.96      0.93      0.94      1000
   transport       0.94      0.95      0.95      1000

    accuracy                           0.94      3000
   macro avg       0.94      0.94      0.94      3000
weighted avg       0.94      0.94      0.94      3000



In [11]:
# Save Homology-Based Label Diffusion Results
# ------------------------------------------
# This is not a trainable model with weights like PyTorch or scikit-learn.
# It is a post-processing step that produces:
# 1. diffused probabilities
# 2. final predictions
# 3. parameters used (alpha and threshold)

import numpy as np
import joblib

# Save predictions
np.save("../models/homology_predictions.npy", y_pred_diffused)

# Save diffused probabilities
np.save("../models/homology_probs.npy", diffused_probs)

# Save parameters used in label diffusion
homology_config = {
    "alpha": alpha,          # e.g., 0.7
    "threshold": threshold   # e.g., 0.95
}

joblib.dump(
    homology_config,
    "../models/homology_config.pkl"
)

# Save complete artifact package
homology_artifacts = {
    "predictions": y_pred_diffused,
    "probabilities": diffused_probs,
    "alpha": alpha,
    "threshold": threshold,
    "label_encoder": label_encoder
}

joblib.dump(
    homology_artifacts,
    "../models/final_homology_model.pkl"
)

print("Homology-Based Label Diffusion artifacts saved successfully!")
print("Saved files:")
print(" - homology_predictions.npy")
print(" - homology_probs.npy")
print(" - homology_config.pkl")
print(" - final_homology_model.pkl")

Homology-Based Label Diffusion artifacts saved successfully!
Saved files:
 - homology_predictions.npy
 - homology_probs.npy
 - homology_config.pkl
 - final_homology_model.pkl
